# 13. 초기 반응 그룹의 만족도(긍정률) 차이 요인 탐색

**분석 목적:** 리뷰 10~49개 구간에서 이미 첫 유저 반응은 확보했지만, 그 반응이 높은 만족도(긍정률 80% 이상)로 이어지는 조건이 무엇인지 탐색한다.

**분석 질문:** 같은 초기 반응 구간(`scale_grade == 'low'`) 안에서도 어떤 장르·가격대·태그가 더 높은 초기 만족도와 연결되는가?

**분석 대상:** `steam_indie_games_graded.csv` 중 `scale_grade == 'low'` 게임만 사용한다.

In [1]:
import ast
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

## 1. 데이터 로드 및 초기 반응 그룹 선별

`steam_indie_games_graded.csv`에서 초기 반응 구간인 `scale_grade == 'low'`(리뷰 10~49개)만 추출한다. 이후 `satisfaction_grade`를 기준으로 `high`와 `mid/low`를 비교해, 같은 노출 수준에서도 만족도가 갈리는 요인을 본다.

In [2]:
DATA_DIR = Path('../../data')
GAMES_PATH = DATA_DIR / 'preprocessed/steam_indie_games_graded.csv'

df_games = pd.read_csv(GAMES_PATH)
df_init = df_games[df_games['scale_grade'] == 'low'].copy()

def parse_list_column(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            return parsed if isinstance(parsed, list) else []
        except Exception:
            return []
    return []

def parse_tag_dict(value):
    if pd.isna(value):
        return []
    if isinstance(value, dict):
        return list(value.keys())
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            parsed = json.loads(value)
            if isinstance(parsed, dict):
                return list(parsed.keys())
            if isinstance(parsed, list):
                return parsed
        except Exception:
            try:
                parsed = ast.literal_eval(value)
                if isinstance(parsed, dict):
                    return list(parsed.keys())
                if isinstance(parsed, list):
                    return parsed
            except Exception:
                return []
    return []

df_init['genres'] = df_init['genres'].apply(parse_list_column)
df_init['tag_list'] = df_init['tags'].apply(parse_tag_dict)
df_init['satisfaction_level'] = df_init['satisfaction_grade'].apply(
    lambda x: '긍정적 (high)' if x == 'high' else '복합/부정적 (mid/low)'
)

print(f"초기 반응 그룹(scale_grade='low') 총 게임 수: {len(df_init):,}개")
print(f"태그 보유 게임 수: {df_init['tag_list'].apply(bool).sum():,}개")

print('\n만족도 수준별 분포:')
dist = df_init['satisfaction_level'].value_counts().rename_axis('satisfaction_level').reset_index(name='game_count')
dist['ratio(%)'] = (dist['game_count'] / len(df_init) * 100).round(1)
display(dist)

초기 반응 그룹(scale_grade='low') 총 게임 수: 4,904개
태그 보유 게임 수: 4,904개

만족도 수준별 분포:


,satisfaction_level,game_count,ratio(%)
0,긍정적 (high),3518,71.7
1,복합/부정적 (mid/low),1386,28.3


**해석:** 현재 초기 반응 구간 게임은 총 `4,904개`이며, 이 중 `긍정적(high)`은 `3,518개(71.7%)`, `복합/부정적(mid/low)`은 `1,386개(28.3%)`다. 즉 리뷰 10~49개 구간에 도달한 게임도 약 3분의 1은 만족도 측면에서 기대를 충분히 충족하지 못한다.

## 2. 장르별 만족도 차이 분석

장르별 게임 수 차이 자체보다, 각 장르 안에서 `긍정적(high)` 비율이 얼마나 높은지 비교한다. 이렇게 해야 '규모가 큰 장르'와 '초기 만족도가 높은 장르'를 분리해서 볼 수 있다.

In [3]:
TARGET_GENRES = ['Action', 'Adventure', 'Casual', 'RPG', 'Simulation', 'Strategy', 'Sports', 'Racing']

df_genre = df_init.explode('genres')
df_genre = df_genre[df_genre['genres'].isin(TARGET_GENRES)].copy()

genre_sat = (
    df_genre.groupby('genres')
    .agg(
        total_games=('appid', 'count'),
        positive_games=('satisfaction_level', lambda s: (s == '긍정적 (high)').sum()),
        negative_games=('satisfaction_level', lambda s: (s == '복합/부정적 (mid/low)').sum()),
    )
)
genre_sat['positive_ratio(%)'] = (genre_sat['positive_games'] / genre_sat['total_games'] * 100).round(1)
genre_sat['negative_ratio(%)'] = (genre_sat['negative_games'] / genre_sat['total_games'] * 100).round(1)
genre_sat = genre_sat.sort_values('positive_ratio(%)', ascending=True).reset_index()

display(genre_sat)

fig = px.bar(
    genre_sat,
    x='positive_ratio(%)',
    y='genres',
    orientation='h',
    text='positive_ratio(%)',
    color='positive_ratio(%)',
    color_continuous_scale='Blues',
    custom_data=['total_games', 'positive_games', 'negative_games'],
    title='장르별 초기 만족도 비율',
    labels={'positive_ratio(%)': '긍정적 비율(%)', 'genres': '장르'}
)
fig.update_traces(
    texttemplate='%{text:.1f}%',
    textposition='outside',
    hovertemplate=(
        '장르=%{y}<br>'
        '긍정적 비율=%{x:.1f}%<br>'
        '총 게임 수=%{customdata[0]:,}<br>'
        '긍정적=%{customdata[1]:,}<br>'
        '복합/부정적=%{customdata[2]:,}<extra></extra>'
    ),
)
fig.update_layout(template='plotly_white', width=900, height=500, coloraxis_showscale=False)
fig.show()

,genres,total_games,positive_games,negative_games,positive_ratio(%),negative_ratio(%)
0,Simulation,1115,626,489,56.1,43.9
1,Racing,178,109,69,61.2,38.8
2,Sports,189,123,66,65.1,34.9
3,RPG,904,600,304,66.4,33.6
4,Adventure,2472,1689,783,68.3,31.7
5,Strategy,945,670,275,70.9,29.1
6,Action,2214,1580,634,71.4,28.6
7,Casual,2330,1721,609,73.9,26.1


**해석:** 현재 데이터에서는 `Simulation`의 초기 만족도 비율이 가장 낮고(`56.1%`), `Casual`이 가장 높다(`73.9%`). 다만 이 결과는 장르 자체의 우열이라기보다, 해당 장르에서 유저가 기대하는 완성도와 실제 제공 경험의 간극이 어디서 크게 발생하는지 보여주는 신호로 읽는 것이 적절하다.

## 3. 가격대별 만족도 차이 분석 (가격 기대치 장벽)

유저는 지불한 가격만큼의 퀄리티를 기대한다. 가격대가 높아질수록 유저의 초기 평가 기준이 얼마나 까다로워지는지 확인한다.

In [4]:
PRICE_BINS = [0, 5, 10, 15, 20, 30, 60, float('inf')]
PRICE_LABELS = ['~$5', '$5~10', '$10~15', '$15~20', '$20~30', '$30~60', '$60+']

price_df = df_init[df_init['price'] > 0].copy()
price_df['price_range'] = pd.cut(
    price_df['price'],
    bins=PRICE_BINS,
    labels=PRICE_LABELS,
    right=False,
    include_lowest=True,
)

price_sat = (
    price_df.groupby('price_range', observed=True)
    .agg(
        total_games=('appid', 'count'),
        negative_games=('satisfaction_level', lambda s: (s == '복합/부정적 (mid/low)').sum()),
        positive_games=('satisfaction_level', lambda s: (s == '긍정적 (high)').sum()),
    )
    .reset_index()
)
price_sat['negative_ratio(%)'] = (price_sat['negative_games'] / price_sat['total_games'] * 100).round(1)
display(price_sat)

fig = px.bar(
    price_sat,
    x='price_range',
    y='negative_ratio(%)',
    text='negative_ratio(%)',
    color='negative_ratio(%)',
    color_continuous_scale='Reds',
    custom_data=['total_games', 'negative_games', 'positive_games'],
    title='가격대별 초기 복합/부정적 평가 비율',
    labels={'negative_ratio(%)': '복합/부정적 비율(%)', 'price_range': '가격대'}
)
fig.update_traces(
    texttemplate='%{text:.1f}%',
    textposition='outside',
    hovertemplate=(
        '가격대=%{x}<br>'
        '복합/부정적 비율=%{y:.1f}%<br>'
        '총 게임 수=%{customdata[0]:,}<br>'
        '복합/부정적=%{customdata[1]:,}<br>'
        '긍정적=%{customdata[2]:,}<extra></extra>'
    ),
)
fig.update_layout(template='plotly_white', width=850, height=450, coloraxis_showscale=False)
fig.show()

,price_range,total_games,negative_games,positive_games,negative_ratio(%)
0,~$5,2751,782,1969,28.4
1,$5~10,1378,371,1007,26.9
2,$10~15,483,115,368,23.8
3,$15~20,191,68,123,35.6
4,$20~30,67,32,35,47.8
5,$30~60,15,6,9,40.0
6,$60+,19,12,7,63.2


**해석:** `$10~15` 구간은 복합/부정적 비율이 상대적으로 낮지만(`23.8%`), `$20` 이상 구간은 비율이 빠르게 높아진다. 특히 `$60+`는 표본 수가 `19개`로 작아 과대해석은 금물이지만, 고가 전략일수록 유저가 기대하는 품질 허들을 넘지 못했을 때 초기 만족도 하락이 더 크게 나타날 가능성을 시사한다.

## 4. 태그 기반 긍정/부정 요인 특이도(Lift) 분석

초기 반응을 얻은 게임들은 `steam_indie_tags.csv`에 태그 데이터가 존재한다. 
'긍정적' 그룹에서 과대표현되는 태그(성공적인 기획 요소)와 '복합적/부정적' 그룹에서 과대표현되는 태그(실망을 유발하기 쉬운 기획 요소)를 추출한다.

In [5]:
GENERAL_TAGS = set(TARGET_GENRES + ['Indie', 'Singleplayer', 'Multiplayer', '2D', '3D'])

def calculate_tag_lift(df, target_level, min_games=30):
    df_exploded = df[['appid', 'satisfaction_level', 'tag_list']].explode('tag_list').dropna(subset=['tag_list']).copy()
    overall_target_ratio = (df['satisfaction_level'] == target_level).mean()

    tag_stats = df_exploded.groupby('tag_list').agg(
        total_games=('appid', 'count'),
        target_games=('satisfaction_level', lambda s: (s == target_level).sum()),
    )
    tag_stats['target_ratio'] = tag_stats['target_games'] / tag_stats['total_games']
    tag_stats['lift'] = tag_stats['target_ratio'] / overall_target_ratio
    tag_stats = tag_stats[(tag_stats['total_games'] >= min_games) & (~tag_stats.index.isin(GENERAL_TAGS))]
    return tag_stats.sort_values(['lift', 'total_games'], ascending=[False, False])

top_positive_tags = calculate_tag_lift(df_init, '긍정적 (high)').head(15).reset_index().rename(columns={'tag_list': 'tag'})
top_negative_tags = calculate_tag_lift(df_init, '복합/부정적 (mid/low)').head(15).reset_index().rename(columns={'tag_list': 'tag'})

display(top_positive_tags[['tag', 'total_games', 'target_ratio', 'lift']].round(3))
display(top_negative_tags[['tag', 'total_games', 'target_ratio', 'lift']].round(3))

fig = go.Figure()
fig.add_trace(go.Bar(
    x=top_positive_tags['lift'],
    y=top_positive_tags['tag'],
    name='긍정적 (high)',
    orientation='h',
    marker_color='#52b788',
    customdata=top_positive_tags[['total_games', 'target_ratio']].values,
    hovertemplate='태그=%{y}<br>Lift=%{x:.2f}<br>게임 수=%{customdata[0]:,}<br>긍정적 비율=%{customdata[1]:.1%}<extra></extra>',
))
fig.add_trace(go.Bar(
    x=top_negative_tags['lift'],
    y=top_negative_tags['tag'],
    name='복합/부정적 (mid/low)',
    orientation='h',
    marker_color='#DD8452',
    customdata=top_negative_tags[['total_games', 'target_ratio']].values,
    hovertemplate='태그=%{y}<br>Lift=%{x:.2f}<br>게임 수=%{customdata[0]:,}<br>복합/부정적 비율=%{customdata[1]:.1%}<extra></extra>',
))
fig.update_layout(
    template='plotly_white',
    width=1000,
    height=700,
    barmode='group',
    title='초기 만족도 그룹별 시그니처 태그 Lift',
    xaxis_title='Lift',
    yaxis_title='태그',
)
fig.show()

,tag,total_games,target_ratio,lift
0,Sokoban,70,0.943,1.314
1,Experimental,40,0.925,1.289
2,Music,50,0.920,1.282
3,Typing,34,0.912,1.271
4,Wholesome,75,0.907,1.264
5,Grid-Based Movement,138,0.906,1.263
6,Cozy,119,0.891,1.242
7,Precision Platformer,292,0.880,1.227
8,Abstract,205,0.878,1.224
9,Otome,40,0.875,1.220


,tag,total_games,target_ratio,lift
0,Mature,35,0.686,2.426
1,Trading,36,0.667,2.359
2,Hentai,35,0.657,2.325
3,Immersive,40,0.625,2.211
4,Hunting,34,0.618,2.185
5,Nudity,52,0.615,2.177
6,Realistic,586,0.565,1.999
7,America,52,0.558,1.973
8,NSFW,30,0.533,1.887
9,Automobile Sim,74,0.527,1.865


**해석:** `Lift > 1`은 해당 태그가 전체 평균보다 특정 만족도 그룹에서 더 자주 나타난다는 뜻이다. 현재 초기 만족도 high 쪽에서는 `Sokoban`, `Wholesome`, `Cozy` 같은 태그가, 복합/부정적 쪽에서는 `Realistic`, `Open World`, `Automobile Sim` 같은 태그가 상대적으로 과대표현된다. 다만 태그는 장르·가격과 함께 작동하므로 단일 원인으로 해석하기보다, 유저 기대치 관리가 까다로운 기획 조합의 신호로 읽는 것이 적절하다.

## 5. 결론 및 인사이트

1. **장르별 기대치 차이:** 초기 반응 구간에서도 장르별 만족도 편차가 뚜렷하다. 특히 `Simulation`처럼 시스템 완성도와 UX 기대치가 높은 장르는 같은 노출 수준에서도 만족도 하락 위험이 더 크다.
2. **가격 허들과 품질 기대감:** `$10~15` 구간은 비교적 안정적이지만, `$20` 이상으로 갈수록 복합/부정적 비율이 높아진다. 고가 전략은 단순 가격 문제가 아니라 유저가 기대하는 콘텐츠 양·완성도·차별성을 동시에 맞춰야 한다.
3. **태그 조합의 시그널:** `Cozy`, `Wholesome`, `Sokoban`처럼 경험이 비교적 명확한 태그는 높은 초기 만족도와 연결되는 반면, `Realistic`, `Open World`, `Automobile Sim`처럼 구현 난이도와 기대치가 높은 태그는 초기 만족도 하락과 함께 나타난다.
4. **실무적 해석:** 리뷰 10~49개 구간은 '노출은 받았지만 평가가 굳어지기 전' 단계다. 이 구간에서 장르·가격·태그 조합이 기대치를 얼마나 안정적으로 충족하는지가 이후 평판 확산의 핵심 변수로 볼 수 있다.